## Code Execution and the Files API

The **code execution tool** lets Claude write and *run* Python code in a secure sandbox on Anthropic's servers. That makes it useful for data analysis, calculations, and creating charts or files, because Claude can check its work by running it.

The **Files API** is how files move in and out:

- You **upload** a file once and get a `file_id`.
- You give that `file_id` to Claude so the sandbox can open the file.
- Files that Claude creates, such as a chart, also get a `file_id`, which you use to **download** them.

### The flow in this notebook

1. Upload `streaming.csv` and get its `file_id`.
2. Ask Claude to analyse the data, giving it the file and the `code_execution` tool.
3. Claude writes and runs code in the sandbox, possibly several times, and creates a plot.
4. Find the plot's `file_id` in the response and download it.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-haiku-5-5"

**What this cell does:** Loads your API key and creates a client with the beta features turned on.

- `Path` (from `pathlib`) is used by the upload helper to read a file's name and extension.
- `load_dotenv()` reads `ANTHROPIC_API_KEY` from your `.env` file.
- `default_headers={"anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"}` adds a header to every request that switches on the code execution tool and the Files API.
- `model = "claude-haiku-5-5"` is the model used for every request.

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    effort=None,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    # Haiku 5.5 thinks adaptively by default; effort sets how hard it thinks
    if effort:
        params["output_config"] = {"effort": effort}

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

**What this cell does:** Defines the helper functions, including the ones for working with files.

- `add_user_message` and `add_assistant_message` add one message to the conversation. The `content` can be a list of blocks, which is how you send text and a file together.
- `chat` sends the conversation to Claude and returns the whole response. `effort` is optional and sets how hard Claude thinks. `max_tokens` is `10000` because thinking and the analysis both use tokens.
- `text_from_message` returns only the text blocks of a response.
- `upload(file_path)` finds the right MIME type for the file's extension (for example `.csv` is `text/csv`), raises an error for an unknown type, and uploads the file with the Files API.
- `list_files`, `get_metadata` and `delete_file` list your files, look up one file's details, and delete one.
- `download_file(id, filename=None)` downloads a file and saves it on your computer. Without a `filename` it uses the file's original name.

In [3]:
file_metadata = upload("streaming.csv")
file_metadata

FileMetadata(id='file_01XgcLWGPZr6oeohvEQTEJ24', created_at=datetime.datetime(2026, 11, 17, 9, 50, 19, 613092, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, scope=None)

**What this cell does:** Uploads `streaming.csv` and shows the details of the uploaded file.

- `upload("streaming.csv")` sends the file to the Files API.
- The result, `file_metadata`, holds the `id` (used in the next cell), the `filename`, the `mime_type`, and the size in bytes.
- `downloadable=False` means you cannot download a file that you uploaded. Only files that Claude creates in the sandbox can be downloaded.

In [4]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

response = chat(
    messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}]
)

response

Message(id='msg_011Ch8Hv92dgsQ4gem79B2Us', container=Container(id='container_016QnBhqW28kDhe5m3kprgdm', expires_at=datetime.datetime(2026, 11, 17, 10, 51, 31, 984427, tzinfo=TzInfo(0))), content=[ThinkingBlock(signature='CAQS8gQKEAgTGAI4AUIIdGhpbmtpbmcSDPlTqWR5JXJwHsJGrBoMwq98lhq20LGnK7exIjC9mfKIIGV3xem+N5suV1ONjpc/2s9Om1kTJylMq0lxlzjTsMCTqjD3WGfV7d12ziYqjwTMjfGn0Iyg3gLT/ZGKxyMI2ZHJQKks5KxqMsMbgCXbfOLCavElM8Rl8N9Of0a1Ra/7+4bJMDWY8RMa1i7shkv7l85t9BrquHeEbN1o4fiXev6a58Zu0lVa3PoaMrts3Cl833ICUt5p+wmf6VoY+BfjuU4tDap2RojLeuNBhvfOPq5xDdDrvNla3tJ2xkSSXAMwmEh+GHFHOZkW59tjzqP3O8CgU5jg8DJuIbZqcKb55KLD1VB3etwMXUET0nNPgv2+WCvenNGk090TNBp3GMI8F5UcdpZTvYz5zYaGoGXAGt61zBhOwAY8KFQiBRGnoMz6uHQOwtL8iGWH1t1FGGSDUm/DIepRr2A5umIPix4Qrq1zUxai4eiiYNkU7lFFTR2ud/j1c17Rar6EyuQPSQGroTQctGZrqY8LEIa65qz4J3urCsAhmG1USP8ORnkW0EnZ0cQpBz1qjxs2k5TZIu/u1QUUZ1Eev16ZZ6fPme90XqtzMveLzgoyDeAui/FYXqxL75o+3Dob5umAfTjTsXg6C/CRSEj082LakiGXf7mGtJWCr+9TPvEfCOGJ55QgmO/ZpgAPJ0gdiVRp3MTLUSnyy20n4maUJ1uWtEOFRwaPlrWK2g4rkpjUS45dOMmeixW

**What this cell does:** Asks Claude to analyse the uploaded CSV with the code execution tool, and keeps the reply in `response`.

- The `container_upload` block gives Claude's sandbox access to the file you uploaded.
- Claude writes and runs Python in the sandbox, possibly several times, and creates a plot.
- `response = chat(...)` saves the whole reply. The next cell needs it to find the plot's file ID.

In [5]:
# Find the files that code execution created (here: the plot)
output_file_ids = []

for block in response.content:
    if block.type == "bash_code_execution_tool_result":
        result = block.content
        if result.type == "bash_code_execution_result":
            for item in result.content:
                if item.type == "bash_code_execution_output":
                    output_file_ids.append(item.file_id)

print(output_file_ids)

# Download the most recent file
download_file(output_file_ids[-1], filename="churn_analysis_comprehensive.png")


['file_01JLxJCufQjg6C8AtkaoFaTd', 'file_01Lz2zDYYZoox8zoToqmrEKF']


**What this cell does:** Finds the plot Claude created and downloads it.

- Each code run comes back as a `bash_code_execution_tool_result` block. When it succeeded, its `content` holds a `bash_code_execution_result`, and any files it created are listed there as `bash_code_execution_output` items, each with a `file_id`.
- The loop collects those IDs into `output_file_ids`. The IDs are different every time you run the notebook, which is why the cell reads them from `response` instead of using a copied ID.
- `download_file(output_file_ids[-1], ...)` saves the last file in the list as `churn_analysis_comprehensive.png` next to the notebook.
- Claude can create more than one file. The saved run listed three IDs. The last one is usually the finished plot, but print the list and check the downloaded picture. If it is not the one you want, use another ID from the list.

> **Note:** If the list is empty, Claude did not create a file on that run. Run the previous cell again, or ask for the plot to be saved as a PNG.